# Plain CNN baseline — the ablation for WaferCNN

Same dataset, same split, same metric as `train_simple.ipynb`. The only thing that
changes is the model and the training recipe, so any difference in score is
attributable to the architecture rather than to the data.

**What this model deliberately does NOT have:**

| WaferCNN has | this baseline |
|---|---|
| residual (skip) connections | plain stacked convs |
| BatchNorm after every conv | none |
| strided conv downsampling | MaxPool |
| global average pooling | flatten + big dense layer |
| dihedral augmentation (8×) | none |
| sqrt inverse-frequency class weights | unweighted loss |
| cosine LR schedule + weight decay | plain Adam |

Run **Runtime → Change runtime type → T4 GPU**. About 4 minutes after the data is cached.

In [ ]:
#@title 1. Data — identical to train_simple.ipynb (reuses the cache if present)
!pip -q install kagglehub

import numpy as np, os, sys, pickle, warnings
warnings.filterwarnings("ignore")

CLASSES = ["Center","Donut","Edge-Loc","Edge-Ring","Local","Random","Scratch","Near-full","None"]
NC, CACHE = 9, "wm811k_64.npz"

if not os.path.exists(CACHE):
    from getpass import getpass
    if not os.environ.get("KAGGLE_KEY"):
        try:
            from google.colab import userdata
            os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
            os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")
        except Exception:
            os.environ["KAGGLE_USERNAME"] = input("Kaggle username: ").strip()
            os.environ["KAGGLE_KEY"]      = getpass("Kaggle key: ").strip()

    import kagglehub, pandas as pd
    from PIL import Image
    from sklearn.model_selection import train_test_split

    pkl = os.path.join(kagglehub.dataset_download("qingyi/wm811k-wafer-map"), "LSWMD.pkl")
    import pandas.core.indexes as _ci
    sys.modules.setdefault("pandas.indexes", _ci)
    sys.modules.setdefault("pandas.indexes.base", _ci.base)
    sys.modules.setdefault("pandas.indexes.numeric", getattr(_ci, "numeric", _ci.base))
    with open(pkl, "rb") as f:
        df = pickle.load(f, encoding="latin1")

    fix = {"none":"None","center":"Center","donut":"Donut","edge-loc":"Edge-Loc",
           "edge-ring":"Edge-Ring","loc":"Local","local":"Local","random":"Random",
           "scratch":"Scratch","near-full":"Near-full","nearfull":"Near-full"}
    def label_of(ft):
        if isinstance(ft, np.ndarray): ft = ft.tolist()
        if not ft or not ft[0]: return None
        s = str(ft[0][0] if isinstance(ft[0], list) else ft[0]).strip()
        return fix.get(s.lower(), s)

    df["label"] = [label_of(v) for v in df["failureType"]]
    lab = df[df["label"].notna()]
    none_m = lab["label"] == "None"
    use = pd.concat([lab[~none_m], lab[none_m].sample(frac=0.25, random_state=42)],
                    ignore_index=True)

    X, y = [], []
    for wm, lb in zip(use["waferMap"], use["label"]):
        if not isinstance(wm, np.ndarray) or wm.size == 0: continue
        X.append(np.array(Image.fromarray(wm.astype(np.uint8), "L")
                          .resize((64, 64), Image.NEAREST), dtype=np.uint8))
        y.append(CLASSES.index(lb))

    X = np.stack(X)[:, None]; y = np.array(y, dtype=np.int64)
    Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
    np.savez_compressed(CACHE, X_train=Xtr, y_train=ytr, X_val=Xva, y_val=yva)

d = np.load(CACHE)
X_train = d["X_train"].astype(np.float32) / 2.0
X_val   = d["X_val"].astype(np.float32)   / 2.0
y_train, y_val = d["y_train"].astype(np.int64), d["y_val"].astype(np.int64)
print("train", X_train.shape, " val", X_val.shape)

## 2 · The model

Three `conv → ReLU → maxpool` stages, then flatten into a dense head. This is the CNN
every textbook opens with — LeNet-5 (LeCun, 1998) has exactly this shape.

In [ ]:
#@title 2. SimpleCNN + shape trace
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


class SimpleCNN(nn.Module):
    def __init__(self, nc=NC):
        super().__init__()
        self.conv1 = nn.Conv2d(1,  16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.relu  = nn.ReLU()
        self.fc1   = nn.Linear(64 * 8 * 8, 128)   # flatten 4096 -> 128
        self.fc2   = nn.Linear(128, nc)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))   # 64 -> 32
        x = self.pool(self.relu(self.conv2(x)))   # 32 -> 16
        x = self.pool(self.relu(self.conv3(x)))   # 16 ->  8
        x = x.flatten(1)
        return self.fc2(self.relu(self.fc1(x)))


# ── shape trace: put this table on a slide ────────────────────────────────────
m, x = SimpleCNN(), torch.zeros(1, 1, 64, 64)
pc = lambda mod: sum(p.numel() for p in mod.parameters())
print(f'\n{"stage":<22}{"output shape":<22}{"params":>10}')
print("-" * 54)
print(f'{"input":<22}{str(tuple(x.shape)):<22}{0:>10,}')
for nm, conv in [("conv1+relu+pool", m.conv1), ("conv2+relu+pool", m.conv2),
                 ("conv3+relu+pool", m.conv3)]:
    x = m.pool(m.relu(conv(x)))
    print(f'{nm:<22}{str(tuple(x.shape)):<22}{pc(conv):>10,}')
x = x.flatten(1);        print(f'{"flatten":<22}{str(tuple(x.shape)):<22}{0:>10,}')
x = m.relu(m.fc1(x));    print(f'{"fc1+relu":<22}{str(tuple(x.shape)):<22}{pc(m.fc1):>10,}')
x = m.fc2(x);            print(f'{"fc2":<22}{str(tuple(x.shape)):<22}{pc(m.fc2):>10,}')
print("-" * 54)
total = pc(m)
print(f'{"TOTAL":<22}{"":<22}{total:>10,}')
print(f'\nfc1 alone is {pc(m.fc1)/total:.1%} of all parameters — '
      f'WaferCNN replaces this layer with global average pooling (0 params).')

### Read the trace before moving on

Two things to notice, both worth a sentence in the presentation:

1. **Total parameters are in the same ballpark as WaferCNN** (615,801). So any score
   difference is *not* explained by model capacity.
2. **One dense layer holds ~95% of them.** `Linear(4096, 128)` memorises absolute
   positions in the feature map. That is where the overfitting comes from, and it is
   exactly what global average pooling removes.

In [ ]:
#@title 3. Train — plain recipe, 15 epochs
import time
from sklearn.metrics import f1_score, accuracy_score

EPOCHS = 15
tl = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
                batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
vl = DataLoader(TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val)),
                batch_size=512, num_workers=2)

torch.manual_seed(0)
model = SimpleCNN().to(DEVICE)
crit  = nn.CrossEntropyLoss()                        # unweighted — deliberate
opt   = torch.optim.Adam(model.parameters(), lr=1e-3)  # no schedule, no weight decay

hist, t0 = [], time.time()
for ep in range(1, EPOCHS + 1):
    model.train()
    for xb, yb in tl:                                 # no augmentation — deliberate
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()

    model.eval(); pr = []
    with torch.no_grad():
        for xb, _ in vl: pr.append(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
    p   = np.concatenate(pr)
    f1  = f1_score(y_val, p, average="macro", labels=range(NC), zero_division=0)
    acc = accuracy_score(y_val, p)
    hist.append((ep, f1, acc))
    print(f"epoch {ep:2d}/{EPOCHS}   macro-F1 {f1:.4f}   accuracy {acc:.4f}")

print(f"\ndone in {time.time()-t0:.0f}s")

In [ ]:
#@title 4. Evaluate + compare against WaferCNN
per   = f1_score(y_val, p, average=None, labels=range(NC), zero_division=0)
never = [CLASSES[i] for i in range(NC) if (p == i).sum() == 0]

# published WaferCNN numbers, from src/models/vision/holdout_results.json
WCNN = {"Center":0.959,"Donut":0.906,"Edge-Loc":0.903,"Edge-Ring":0.984,"Local":0.834,
        "Random":0.902,"Scratch":0.861,"Near-full":0.978,"None":0.983}
WCNN_MACRO, WCNN_ACC = 0.9232, 0.9617

print(f'{"class":<12}{"support":>9}{"simple":>9}{"WaferCNN":>10}{"delta":>9}')
print("-" * 49)
for i, c in enumerate(CLASSES):
    print(f'{c:<12}{(y_val==i).sum():>9}{per[i]:>9.3f}{WCNN[c]:>10.3f}{per[i]-WCNN[c]:>+9.3f}')
print("-" * 49)
print(f'{"MACRO-F1":<12}{"":>9}{f1:>9.4f}{WCNN_MACRO:>10.4f}{f1-WCNN_MACRO:>+9.4f}')
print(f'{"ACCURACY":<12}{"":>9}{acc:>9.4f}{WCNN_ACC:>10.4f}{acc-WCNN_ACC:>+9.4f}')

print(f'\nNEVER PREDICTS: {never or "none — every class gets used"}')

# the number that makes accuracy meaningless here
base = (y_val == CLASSES.index("None")).mean()
print(f'\n"always predict None" baseline:  accuracy {base:.4f},  macro-F1 '
      f'{f1_score(y_val, np.full_like(y_val, CLASSES.index("None")), average="macro", labels=range(NC), zero_division=0):.4f}')
print("Quote macro-F1. Accuracy near 0.59 means the model learned nothing.")

## What to say about the result

Look at **where** the gap is, not just the headline number. The plain CNN should stay
close on `None` and `Edge-Ring` (thousands of examples, distinctive shapes) and fall
behind on `Near-full`, `Donut`, `Random` and `Scratch` — the classes with a few dozen
to a few hundred examples.

That pattern is the argument, and each part of it maps to one missing ingredient:

| Gap you observe | Caused by |
|---|---|
| rare classes much worse | unweighted loss — the model has little incentive to get 22 Near-full wafers right |
| rotated/mirrored patterns confused | no dihedral augmentation, so it saw each wafer in one orientation only |
| val score plateaus then drifts down | 524k parameters in `fc1` with no regularisation |
| needs more epochs to stabilise | no BatchNorm, so a larger learning rate diverges |

**If the plain CNN lands close to WaferCNN, say so.** The honest reading is that the
architecture modifications bought less than expected on this task, which is itself a
finding — 64×64 binary maps are an easy input, and there is a ceiling that extra
capacity does not move. Do not tune this baseline until it loses; a deliberately
crippled comparison proves nothing.